In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone per rule
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to timezone-aware ISO strings (UTC+8)
def ts_to_tz_iso(s):
    # s is a Series of integer seconds
    return pd.to_datetime(s, unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 1) metric.csv summary grouped by cmdb_id and kpi_name
g = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_stats = g.agg(
    count='count',
    min='min',
    p50=lambda x: x.quantile(0.5),
    p95=lambda x: x.quantile(0.95),
    max='max'
).reset_index()

ts_bounds = metric_df.groupby(['cmdb_id', 'kpi_name'])['timestamp'].agg(ts_earliest='min', ts_latest='max').reset_index()
metric_summary = metric_stats.merge(ts_bounds, on=['cmdb_id','kpi_name'])
metric_summary['ts_earliest'] = ts_to_tz_iso(metric_summary['ts_earliest'])
metric_summary['ts_latest'] = ts_to_tz_iso(metric_summary['ts_latest'])

metric_summary = metric_summary.sort_values('count', ascending=False).head(50)
metric_summary = metric_summary[['cmdb_id','kpi_name','count','min','p50','p95','max','ts_earliest','ts_latest']]

# 2) trace.csv summary grouped by cmdb_id and trace_name
g = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_stats = g.agg(
    count='count',
    min='min',
    p50=lambda x: x.quantile(0.5),
    p95=lambda x: x.quantile(0.95),
    max='max'
).reset_index()

ts_bounds = trace_df.groupby(['cmdb_id', 'trace_name'])['timestamp'].agg(ts_earliest='min', ts_latest='max').reset_index()
trace_summary = trace_stats.merge(ts_bounds, on=['cmdb_id','trace_name'])
trace_summary['ts_earliest'] = ts_to_tz_iso(trace_summary['ts_earliest'])
trace_summary['ts_latest'] = ts_to_tz_iso(trace_summary['ts_latest'])

trace_summary = trace_summary.sort_values('count', ascending=False).head(50)
trace_summary = trace_summary[['cmdb_id','trace_name','count','min','p50','p95','max','ts_earliest','ts_latest']]

# 3) log.csv summary grouped by cmdb_id and log_name
g = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_stats = g.agg(
    count='count',
    min='min',
    p50=lambda x: x.quantile(0.5),
    p95=lambda x: x.quantile(0.95),
    max='max'
).reset_index()

ts_bounds = log_df.groupby(['cmdb_id', 'log_name'])['timestamp'].agg(ts_earliest='min', ts_latest='max').reset_index()
log_summary = log_stats.merge(ts_bounds, on=['cmdb_id','log_name'])
log_summary['ts_earliest'] = ts_to_tz_iso(log_summary['ts_earliest'])
log_summary['ts_latest'] = ts_to_tz_iso(log_summary['ts_latest'])

log_summary = log_summary.sort_values('count', ascending=False).head(50)
log_summary = log_summary[['cmdb_id','log_name','count','min','p50','p95','max','ts_earliest','ts_latest']]

# 4) error_logs.csv aggregates per cmdb_id and sample earliest messages
error_agg = error_df.groupby('cmdb_id')['timestamp'].agg(count='count', ts_earliest='min', ts_latest='max').reset_index()
error_agg['ts_earliest'] = ts_to_tz_iso(error_agg['ts_earliest'])
error_agg['ts_latest'] = ts_to_tz_iso(error_agg['ts_latest'])
error_agg = error_agg[['cmdb_id','count','ts_earliest','ts_latest']]

# Up to 5 sample messages with earliest timestamps across the whole file
error_samples = error_df.sort_values('timestamp', ascending=True).head(5).copy()
error_samples['timestamp'] = ts_to_tz_iso(error_samples['timestamp'])
error_samples = error_samples[['timestamp','cmdb_id','message']]

# Display compact summaries (variables returned for IPython)
metric_summary, trace_summary, log_summary, error_agg, error_samples

```
Out[1]:
```
```python
# Build a concise plain-English summary using existing summary variables from the previous run.
# We rely on metric_summary, trace_summary, log_summary, error_agg, error_samples being present.

def short(x, n=60):
    s = str(x)
    return s if len(s) <= n else s[:n-3] + '...'

# Metric summary: top 5 groups (already sorted by count)
metric_top5 = metric_summary.head(5)[['cmdb_id','kpi_name','count','ts_earliest','ts_latest']].to_records(index=False)
metric_count_groups = metric_summary.shape[0]

# Trace summary: counts and potential epoch timestamps
trace_total_groups = trace_summary.shape[0]
trace_epoch_count = trace_summary['ts_earliest'].astype(str).str.startswith('1970').sum()
trace_top5 = trace_summary.head(5)[['cmdb_id','trace_name','count','ts_earliest','ts_latest']].to_records(index=False)

# Log summary: top 5 groups
log_total_groups = log_summary.shape[0]
log_top5 = log_summary.head(5)[['cmdb_id','log_name','count','p50','p95','ts_earliest','ts_latest']].to_records(index=False)

# Error logs summary
error_present = not error_agg.empty
if error_present:
    error_counts = error_agg.copy()
    error_sample_rows = error_samples.head(5).to_records(index=False)
else:
    error_counts = None
    error_sample_rows = ()

# Compose a compact plain-English summary string
summary_lines = []
summary_lines.append("Summary of telemetry aggregates (compact):")

# metric
summary_lines.append(f"- metric.csv: {metric_count_groups} groups (showing up to 50). Top groups by count:")
for r in metric_top5:
    cmdb,kpi,cnt,te,tl = r
    summary_lines.append(f"    • {cmdb} | {short(kpi)} | count={int(cnt)} | {te} → {tl}")

# trace
summary_lines.append(f"- trace.csv: {trace_total_groups} groups. {trace_epoch_count} groups have earliest ts in 1970 (possible epoch/base timestamp). Top groups:")
for r in trace_top5:
    cmdb,name,cnt,te,tl = r
    summary_lines.append(f"    • {cmdb} | {short(name)} | count={int(cnt)} | {te}")

# log
summary_lines.append(f"- log.csv: {log_total_groups} groups. Top groups:")
for r in log_top5:
    cmdb,name,cnt,p50,p95,te,tl = r
    summary_lines.append(f"    • {cmdb} | {short(name)} | count={int(cnt)} | p50={float(p50):.3f} p95={float(p95):.3f} | {te} → {tl}")

# error logs
if error_present:
    summary_lines.append(f"- error_logs.csv: per-cmdb aggregates present ({len(error_counts)} cmdb_ids). Earliest sample messages (up to 5):")
    for ts,cmdb,msg in error_sample_rows:
        summary_lines.append(f"    • {ts} | {cmdb} | {short(msg,120)}")
else:
    summary_lines.append("- error_logs.csv: no error log records found in this dataset.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           min           p50           p95           max               ts_earliest                 ts_latest
55        MG01                             OSLinux-CPU_CPU_CPUWio     31  4.991500e-02  1.914190e-01  6.262730e-01  1.247600e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.800000e+01  7.300000e+01  7.600000e+01  7.600000e+01  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
1119  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  1.380000e+02  1.380000e+02  1.380000e+02  1.380000e+02  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
1129  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
1136  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  4.000000e+00  9.000000e+00  1.000000e+01  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
...        ...                                                ...    ...           ...           ...           ...           ...                       ...                       ...
877   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.721851e+06  2.728332e+06  2.734769e+06  2.735524e+06  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
878   Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
879   Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
846   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
119       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800

[50 rows x 9 columns],       cmdb_id                         trace_name  count          min          p50          p95          max               ts_earliest                 ts_latest
263  dockerB2            trace.to_MG02.row_count      1  5256.000000  5256.000000  5256.000000  5256.000000  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
0        IG01      trace.from_root.duration_mean      1     0.347255     0.347255     0.347255     0.347255  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
1        IG01       trace.from_root.duration_p95      1     1.076500     1.076500     1.076500     1.076500  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
2        IG01          trace.from_root.row_count      1    47.000000    47.000000    47.000000    47.000000  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
3        IG01           trace.self.duration_mean      1     0.553114     0.553114     0.553114     0.553114  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
..        ...                                ...    ...          ...          ...          ...          ...                       ...                       ...
69       MG01    trace.to_dockerA2.duration_mean      1     0.172063     0.172063     0.172063     0.172063  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
70       MG01     trace.to_dockerA2.duration_p95      1     1.031000     1.031000     1.031000     1.031000  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
71       MG01        trace.to_dockerA2.row_count      1  4731.000000  4731.000000  4731.000000  4731.000000  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
56       MG01      trace.from_dockerB1.row_count      1  4823.000000  4823.000000  4823.000000  4823.000000  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
57       MG01  trace.from_dockerB2.duration_mean      1     0.155119     0.155119     0.155119     0.155119  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800

[50 rows x 9 columns],      cmdb_id          log_name  count    min     p50        p95      max               ts_earliest                 ts_latest
9   Tomcat02     log.row_count     31  13.00  449.00   532.0000   581.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
7   Tomcat02   log.error_count     31   0.00    0.00     0.0000     0.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
6   Tomcat01     log.row_count     31   8.00  461.00   532.5000   555.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
4   Tomcat01   log.error_count     31   0.00    0.00     0.0000     0.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
18  apache02   log.error_count     31   0.00    0.00     0.0000     0.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
17  apache01     log.row_count     31  15.00  919.00  1041.5000  1119.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
19  apache02     log.row_count     31  20.00  929.00  1030.5000  1082.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
16  apache01   log.error_count     31   0.00    0.00     0.0000     0.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
10  Tomcat03   log.error_count     31   0.00    0.00     0.0000     0.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
15  Tomcat04     log.row_count     31  13.00  455.00   542.0000   555.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
13  Tomcat04   log.error_count     31   0.00    0.00     0.0000     0.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
12  Tomcat03     log.row_count     31   6.00  452.00   512.0000   524.00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
14  Tomcat04  log.gc_pause_max     24   0.02    0.06     0.0900     0.11  2021-03-23 06:00:00+0800  2021-03-23 06:29:00+0800
8   Tomcat02  log.gc_pause_max     22   0.02    0.07     0.0900     0.10  2021-03-23 06:01:00+0800  2021-03-23 06:29:00+0800
11  Tomcat03  log.gc_pause_max     22   0.02    0.07     0.0995     0.11  2021-03-23 06:00:00+0800  2021-03-23 06:29:00+0800
5   Tomcat01  log.gc_pause_max     21   0.02    0.06     0.0800     0.08  2021-03-23 06:01:00+0800  2021-03-23 06:29:00+0800
2    Mysql02   log.error_count      2   0.00    0.00     0.0000     0.00  2021-03-23 06:07:00+0800  2021-03-23 06:10:00+0800
3    Mysql02     log.row_count      2   1.00    1.00     1.0000     1.00  2021-03-23 06:07:00+0800  2021-03-23 06:10:00+0800
0    Mysql01   log.error_count      1   0.00    0.00     0.0000     0.00  2021-03-23 06:10:00+0800  2021-03-23 06:10:00+0800
1    Mysql01     log.row_count      1   1.00    1.00     1.0000     1.00  2021-03-23 06:10:00+0800  2021-03-23 06:10:00+0800, Empty DataFrame
Columns: [cmdb_id, count, ts_earliest, ts_latest]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids (known possible root-cause components)
targets = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Filter metric_df to only target components
metric_targets = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

# Ensure timestamp is integer
metric_targets['timestamp'] = metric_targets['timestamp'].astype('int64')

# 1) Group by cmdb_id and kpi_name and compute required aggregates using full series
g = metric_targets.groupby(['cmdb_id','kpi_name'])['value']
stats = g.agg(
    count='count',
    min_value='min',
    p95=lambda x: float(x.quantile(0.95)),
    max_value='max'
).reset_index()

ts_bounds = metric_targets.groupby(['cmdb_id','kpi_name'])['timestamp'].agg(ts_earliest='min', ts_latest='max').reset_index()

grouped = stats.merge(ts_bounds, on=['cmdb_id','kpi_name'])

# Convert timestamps to UTC+8 ISO strings for display
grouped['ts_earliest_iso'] = pd.to_datetime(grouped['ts_earliest'], unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
grouped['ts_latest_iso'] = pd.to_datetime(grouped['ts_latest'], unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Select and rename columns as requested
group_summary = grouped[['cmdb_id','kpi_name','count','min_value','p95','max_value','ts_earliest_iso','ts_latest_iso']].copy()
group_summary.columns = ['cmdb_id','kpi_name','count','min','p95','max','ts_earliest','ts_latest']

# Sort by count desc and keep up to top 50
group_summary = group_summary.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# 2) Find groups that cover the incident window:
# incident window in UTC+8: start '2021-03-23 14:00:00+0800', end '2021-03-23 14:30:00+0800'
start_ts = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz).timestamp()  # float seconds
end_ts = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz).timestamp()

# Use the numeric ts_earliest/ts_latest from 'grouped' (which are in seconds)
cover_mask = (grouped['ts_earliest'] <= start_ts) & (grouped['ts_latest'] >= end_ts)
covering = grouped.loc[cover_mask, ['cmdb_id','kpi_name','count','ts_earliest','ts_latest']].copy()

# Convert to ISO strings for display
covering['ts_earliest'] = pd.to_datetime(covering['ts_earliest'], unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
covering['ts_latest'] = pd.to_datetime(covering['ts_latest'], unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Keep up to 20 rows and reset index
covering_groups = covering.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Display compact results (two variables)
group_summary, covering_groups

```
Out[2]:
```
Summary (based on metric.csv for the requested components):

- I computed per (cmdb_id, kpi_name) aggregates (count, global P95, min, max, earliest and latest timestamp) and returned up to 50 groups sorted by count.
- Top groups (examples from the top results):
  - Tomcat04 | Tomcat-Threads..._MaxThreads... — count=31, p95=500, min=500, max=500, ts_earliest=2021-03-23 06:00:00+0800 → ts_latest=2021-03-23 06:30:00+0800
  - IG01     | JVM-Memory..._HeapMemoryMax — count=31, p95≈2.040136e+09, min≈2.040136e+09, max≈2.040136e+09, ts_earliest=2021-03-23 06:00:00+0800 → ts_latest=2021-03-23 06:30:00+0800
  - Tomcat04 | Tomcat-Threads..._CurrentThreads — count=31, p95=10, min=0, max=10, ts_earliest=2021-03-23 06:00:00+0800 → ts_latest=2021-03-23 06:30:00+0800
  - IG01     | JVM-Memory..._HeapMemoryUsed — count=31, p95≈8.898162e+08, min≈5.648289e+07, max≈8.898162e+08, ts_earliest=2021-03-23 06:00:00+0800 → ts_latest=2021-03-23 06:30:00+0800
  - (Many other groups appear with count=31; full top-50 was returned in the earlier run.)

- Incident-window coverage check (window: 2021-03-23 14:00:00+0800 → 2021-03-23 14:30:00+0800):
  - No metric groups from the specified components cover that incident window. The groups in the summary have earliest/latest timestamps around 2021-03-23 06:00:00+0800 → 2021-03-23 06:30:00+0800, so they do not span 14:00–14:30+0800.

If you want, I can:
- Show the full top-50 table (or a specific subset) with numeric p95 values for more groups, or
- Re-run the coverage check if you want a different incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count           min           p95           max               ts_earliest                 ts_latest
0   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
2   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  9.000000e+00  1.000000e+01  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
3       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  5.648289e+07  8.858267e+08  8.898162e+08  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
4       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.867203e+07  5.867203e+07  5.867203e+07  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
..       ...                                                ...    ...           ...           ...           ...                       ...                       ...
45  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  1.700000e+01  2.100000e+01  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
46  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
47  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
48  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
49  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800

[50 rows x 8 columns],      cmdb_id                                           kpi_name  count               ts_earliest                 ts_latest
0   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
2       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
4   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
5       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
6   Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
7       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
8       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
9       MG02           JVM-Threads_7779_JVM_ThreadCount_Threads     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
10      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
11      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
12      MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
13      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
14      MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
15  Tomcat04                        OSLinux-CPU_CPU_CPUUserTime     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
16  Tomcat04                         OSLinux-CPU_CPU_CPUSysTime     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
17      MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
18      MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
19  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing dataframes if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure timestamp is integer seconds
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('int64')

# Incident window in UTC+8 (Asia/Shanghai)
start_ts = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz).timestamp()
end_ts   = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz).timestamp()

# Helper to convert epoch seconds to tz-aware ISO string
def to_iso_s(series):
    return pd.to_datetime(series, unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 1) File-level summaries: total count, earliest ts, latest ts (full series)
summaries = []
for name, df in [('metric.csv', metric_df), ('trace.csv', trace_df), ('log.csv', log_df), ('error_logs.csv', error_df)]:
    if df.shape[0] == 0:
        summaries.append((name, 0, None, None))
    else:
        earliest = int(df['timestamp'].min())
        latest = int(df['timestamp'].max())
        summaries.append((name, df.shape[0], to_iso_s(pd.Series(earliest)).iloc[0], to_iso_s(pd.Series(latest)).iloc[0]))

file_summaries = pd.DataFrame(summaries, columns=['file','total_count','ts_earliest','ts_latest'])

# 2) Extract up to 20 rows per file within the incident window (inclusive)
mask_metric = (metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)
metric_window = metric_df.loc[mask_metric, ['timestamp','cmdb_id','kpi_name','value']].copy().head(20)
if not metric_window.empty:
    metric_window['timestamp'] = to_iso_s(metric_window['timestamp'])

mask_trace = (trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)
trace_window = trace_df.loc[mask_trace, ['timestamp','cmdb_id','trace_name','value']].copy().head(20)
if not trace_window.empty:
    trace_window['timestamp'] = to_iso_s(trace_window['timestamp'])

mask_log = (log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)
log_window = log_df.loc[mask_log, ['timestamp','cmdb_id','log_name','value']].copy().head(20)
if not log_window.empty:
    log_window['timestamp'] = to_iso_s(log_window['timestamp'])

mask_error = (error_df['timestamp'] >= start_ts) & (error_df['timestamp'] <= end_ts)
error_window = error_df.loc[mask_error, ['timestamp','cmdb_id','message']].copy().head(20)
if not error_window.empty:
    error_window['timestamp'] = to_iso_s(error_window['timestamp'])

# Return compact outputs (use variable names for IPython display)
file_summaries, metric_window, trace_window, log_window, error_window

```
Out[3]:
```
# Build a compact plain-English summary string using existing variables from the previous run.
# Reuse variables: file_summaries, metric_window, trace_window, log_window, error_window

# Determine counts of returned window rows (they may be empty DataFrames)
metric_rows = 0 if 'metric_window' not in globals() or metric_window.empty else metric_window.shape[0]
trace_rows  = 0 if 'trace_window' not in globals() or trace_window.empty else trace_window.shape[0]
log_rows    = 0 if 'log_window' not in globals() or log_window.empty else log_window.shape[0]
error_rows  = 0 if 'error_window' not in globals() or error_window.empty else error_window.shape[0]

# Extract file-level summaries
def fs_val(fname, col):
    row = file_summaries.loc[file_summaries['file'] == fname]
    if row.empty:
        return None
    return row.iloc[0][col]

summary_lines = []
summary_lines.append("Compact telemetry file summary (full-series min/max; incident-window sampling):")
summary_lines.append("")
# metric.csv
summary_lines.append(f"- metric.csv: total rows = {int(fs_val('metric.csv','total_count'))}, earliest = {fs_val('metric.csv','ts_earliest')}, latest = {fs_val('metric.csv','ts_latest')}.")
summary_lines.append(f"  • Rows returned within requested window (limited to 20): {metric_rows}. Example cmdb_ids in those rows: " +
                     (", ".join(sorted(metric_window['cmdb_id'].unique()[:5])) if metric_rows>0 else "none"))
summary_lines.append("")
# trace.csv
summary_lines.append(f"- trace.csv: total rows = {int(fs_val('trace.csv','total_count'))}, earliest = {fs_val('trace.csv','ts_earliest')}, latest = {fs_val('trace.csv','ts_latest')}.")
summary_lines.append(f"  • Rows returned within requested window (limited to 20): {trace_rows}.")
summary_lines.append("")
# log.csv
summary_lines.append(f"- log.csv: total rows = {int(fs_val('log.csv','total_count'))}, earliest = {fs_val('log.csv','ts_earliest')}, latest = {fs_val('log.csv','ts_latest')}.")
summary_lines.append(f"  • Rows returned within requested window (limited to 20): {log_rows}. Example cmdb_ids in those rows: " +
                     (", ".join(sorted(log_window['cmdb_id'].unique()[:8])) if log_rows>0 else "none"))
summary_lines.append("")
# error_logs.csv
err_total = fs_val('error_logs.csv','total_count')
summary_lines.append(f"- error_logs.csv: total rows = {0 if err_total is None else int(err_total)}.")
summary_lines.append(f"  • Rows returned within requested window (limited to 20): {error_rows}.")
summary_lines.append("")
summary_lines.append("Notes:")
summary_lines.append("- Timestamps shown are in UTC+8 (Asia/Shanghai).")
summary_lines.append("- Returned per-file window rows were limited to 20 each and are available as variables metric_window, trace_window, log_window, error_window for inspection.")
summary = "\n".join(summary_lines)

# Display the summary string (compact)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  total_count               ts_earliest                 ts_latest
0      metric.csv        21913  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
1       trace.csv          264  1970-01-19 17:01:00+0800  1970-01-19 17:01:00+0800
2         log.csv          467  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800
3  error_logs.csv            0                      None                      None,                    timestamp cmdb_id                                           kpi_name         value
0   2021-03-23 06:00:00+0800    IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1   2021-03-23 06:00:00+0800    IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  6.777046e+08
2   2021-03-23 06:00:00+0800    IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.867203e+07
3   2021-03-23 06:00:00+0800    IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  4.699000e-01
4   2021-03-23 06:00:00+0800    IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.500000e+01
5   2021-03-23 06:00:00+0800    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1.700000e+01
6   2021-03-23 06:00:00+0800    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  1.000000e+00
7   2021-03-23 06:00:00+0800    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  1.470000e+02
8   2021-03-23 06:00:00+0800    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  0.000000e+00
9   2021-03-23 06:00:00+0800    IG01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.838620e+01
10  2021-03-23 06:00:00+0800    IG01         OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  0.000000e+00
11  2021-03-23 06:00:00+0800    IG01       OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00
12  2021-03-23 06:00:00+0800    IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
13  2021-03-23 06:00:00+0800    IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  4.423704e+08
14  2021-03-23 06:00:00+0800    IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.854114e+07
15  2021-03-23 06:00:00+0800    IG02          JVM-Operating System_7778_JVM_JVM_CPULoad  5.405000e-01
16  2021-03-23 06:00:00+0800    IG02           JVM-Threads_7778_JVM_ThreadCount_Threads  7.500000e+01
17  2021-03-23 06:00:00+0800    IG02                         OSLinux-CPU_CPU_CPUCpuUtil  2.640530e+01
18  2021-03-23 06:00:00+0800    IG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01
19  2021-03-23 06:00:00+0800    MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [],                    timestamp   cmdb_id          log_name   value
0   2021-03-23 06:00:00+0800  Tomcat01   log.error_count    0.00
1   2021-03-23 06:00:00+0800  Tomcat01     log.row_count  424.00
2   2021-03-23 06:00:00+0800  Tomcat02   log.error_count    0.00
3   2021-03-23 06:00:00+0800  Tomcat02     log.row_count  414.00
4   2021-03-23 06:00:00+0800  Tomcat03   log.error_count    0.00
5   2021-03-23 06:00:00+0800  Tomcat03  log.gc_pause_max    0.08
6   2021-03-23 06:00:00+0800  Tomcat03     log.row_count  449.00
7   2021-03-23 06:00:00+0800  Tomcat04   log.error_count    0.00
8   2021-03-23 06:00:00+0800  Tomcat04  log.gc_pause_max    0.09
9   2021-03-23 06:00:00+0800  Tomcat04     log.row_count  414.00
10  2021-03-23 06:00:00+0800  apache01   log.error_count    0.00
11  2021-03-23 06:00:00+0800  apache01     log.row_count  849.00
12  2021-03-23 06:00:00+0800  apache02   log.error_count    0.00
13  2021-03-23 06:00:00+0800  apache02     log.row_count  843.00
14  2021-03-23 06:01:00+0800  Tomcat01   log.error_count    0.00
15  2021-03-23 06:01:00+0800  Tomcat01  log.gc_pause_max    0.07
16  2021-03-23 06:01:00+0800  Tomcat01     log.row_count  479.00
17  2021-03-23 06:01:00+0800  Tomcat02   log.error_count    0.00
18  2021-03-23 06:01:00+0800  Tomcat02  log.gc_pause_max    0.06
19  2021-03-23 06:01:00+0800  Tomcat02     log.row_count  468.00, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamp is integer
for df in (metric_df, log_df, trace_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('int64')

# Helper: convert epoch seconds to tz-aware datetime (Asia/Shanghai)
def to_dt_series(ts_series):
    return pd.to_datetime(ts_series, unit='s').dt.tz_localize(tz)

def fmt_dt_single(ts):
    return ts.strftime('%Y-%m-%d %H:%M:%S%z')

# 1) Hourly distribution for metric.csv
metric_dt = to_dt_series(metric_df['timestamp'])
metric_hour_bucket = metric_dt.dt.floor('H').dt.strftime('%Y-%m-%d %H:00:00%z')
metric_hourly = metric_df.copy()
metric_hourly['hour_bucket'] = metric_hour_bucket
metric_hourly = metric_hourly.groupby('hour_bucket', as_index=False).size().rename(columns={'size':'row_count'})
metric_hourly = metric_hourly.sort_values('hour_bucket', ascending=True).head(50)

# Hourly distribution for log.csv
log_dt = to_dt_series(log_df['timestamp'])
log_hour_bucket = log_dt.dt.floor('H').dt.strftime('%Y-%m-%d %H:00:00%z')
log_hourly = log_df.copy()
log_hourly['hour_bucket'] = log_hour_bucket
log_hourly = log_hourly.groupby('hour_bucket', as_index=False).size().rename(columns={'size':'row_count'})
log_hourly = log_hourly.sort_values('hour_bucket', ascending=True).head(50)

# 2) Distinct dates and file-level summary for metric.csv and log.csv
def file_date_summary(df, name):
    if df.shape[0] == 0:
        return pd.DataFrame([{
            'file': name,
            'total_count': 0,
            'ts_earliest': None,
            'ts_latest': None,
            'distinct_dates': ''
        }])
    dt = to_dt_series(df['timestamp'])
    earliest_ts = dt.min()  # Timestamp
    latest_ts = dt.max()
    earliest = fmt_dt_single(earliest_ts)
    latest = fmt_dt_single(latest_ts)
    distinct_dates = sorted(dt.dt.strftime('%Y-%m-%d').unique())
    if len(distinct_dates) > 10:
        distinct_dates_str = ", ".join(distinct_dates[:10]) + ", ..."
    else:
        distinct_dates_str = ", ".join(distinct_dates)
    return pd.DataFrame([{
        'file': name,
        'total_count': int(df.shape[0]),
        'ts_earliest': earliest,
        'ts_latest': latest,
        'distinct_dates': distinct_dates_str
    }])

metric_summary = file_date_summary(metric_df, 'metric.csv')
log_summary = file_date_summary(log_df, 'log.csv')

# 3) For trace.csv: unique timestamp values present (up to 20 distinct values), formatted in UTC+8
if trace_df.shape[0] == 0:
    trace_timestamps = pd.DataFrame(columns=['timestamp_iso']).head(0)
else:
    trace_dt = to_dt_series(trace_df['timestamp'])
    unique_ts = pd.Series(trace_dt.dt.strftime('%Y-%m-%d %H:%M:%S%z').unique()).sort_values().reset_index(drop=True)
    trace_timestamps = unique_ts.head(20).to_frame(name='timestamp_iso')

# Return compact results
metric_hourly, log_hourly, metric_summary, log_summary, trace_timestamps

```
Out[4]:
```
```python
summary = (
    "Summary of results (plain English):\n\n"
    "1) metric.csv\n"
    "   - Total rows: 21,913\n"
    "   - Earliest timestamp (full series): 2021-03-23 06:00:00+0800\n"
    "   - Latest timestamp (full series):   2021-03-23 06:30:00+0800\n"
    "   - Hourly distribution (up to 50 rows): only one hour bucket present: 2021-03-23 06:00:00+0800 with 21,913 rows\n    - Distinct dates present: 2021-03-23\n\n"
    "2) log.csv\n"
    "   - Total rows: 467\n"
    "   - Earliest timestamp (full series): 2021-03-23 06:00:00+0800\n"
    "   - Latest timestamp (full series):   2021-03-23 06:30:00+0800\n"
    "   - Hourly distribution (up to 50 rows): only one hour bucket present: 2021-03-23 06:00:00+0800 with 467 rows\n    - Distinct dates present: 2021-03-23\n\n"
    "3) trace.csv\n"
    "   - Unique timestamp samples (up to 20): observed value: 1970-01-19 17:01:00+0800\n"
    "     -> This indicates the trace timestamps are epoch-like (1970) rather than 2021 timestamps.\n\n"
    "Notes/implication:\n"
    " - Both metric and log data in these files are concentrated in the 2021-03-23 06:00 hour (06:00–06:30+0800), not the 14:00–14:30 window.\n"
    " - Trace timestamps appear to be epoch-like (1970), so treat them as likely non-2021 timestamps or needing correction.\n"
    " - error_logs.csv contained no rows in this dataset (no timestamps/messages to report).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                hour_bucket  row_count
0  2021-03-23 06:00:00+0800      21913,                 hour_bucket  row_count
0  2021-03-23 06:00:00+0800        467,          file  total_count               ts_earliest                 ts_latest distinct_dates
0  metric.csv        21913  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800     2021-03-23,       file  total_count               ts_earliest                 ts_latest distinct_dates
0  log.csv          467  2021-03-23 06:00:00+0800  2021-03-23 06:30:00+0800     2021-03-23,               timestamp_iso
0  1970-01-19 17:01:00+0800)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing dataframes if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure timestamp column is integer seconds
for df in (metric_df, log_df, trace_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('int64')

# Time window specified in UTC
tz_utc = pytz.UTC
tz_local = pytz.timezone('Asia/Shanghai')  # per rule: display in UTC+8

start_utc = pd.Timestamp('2021-03-23 06:00:00', tz=tz_utc)
end_utc   = pd.Timestamp('2021-03-23 06:30:00', tz=tz_utc)

start_ts = int(start_utc.timestamp())
end_ts = int(end_utc.timestamp())

# Helper to format epoch seconds into Asia/Shanghai ISO strings
def format_ts_as_shanghai(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz_local).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Filter and sample for each file
# metric.csv
metric_mask = (metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)
metric_count = int(metric_mask.sum())
metric_samples = metric_df.loc[metric_mask, ['timestamp','cmdb_id','kpi_name','value']].head(10).copy()
if not metric_samples.empty:
    metric_samples['timestamp'] = format_ts_as_shanghai(metric_samples['timestamp'])

# log.csv
log_mask = (log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)
log_count = int(log_mask.sum())
log_samples = log_df.loc[log_mask, ['timestamp','cmdb_id','log_name','value']].head(10).copy()
if not log_samples.empty:
    log_samples['timestamp'] = format_ts_as_shanghai(log_samples['timestamp'])

# trace.csv
trace_mask = (trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)
trace_count = int(trace_mask.sum())
trace_samples = trace_df.loc[trace_mask, ['timestamp','cmdb_id','trace_name','value']].head(10).copy()
if not trace_samples.empty:
    trace_samples['timestamp'] = format_ts_as_shanghai(trace_samples['timestamp'])

# error_logs.csv
error_mask = (error_df['timestamp'] >= start_ts) & (error_df['timestamp'] <= end_ts)
error_count = int(error_mask.sum())
error_samples = error_df.loc[error_mask, ['timestamp','cmdb_id','message']].head(10).copy()
if not error_samples.empty:
    error_samples['timestamp'] = format_ts_as_shanghai(error_samples['timestamp'])

# Compact counts summary DataFrame
window_counts = pd.DataFrame([
    {'file':'metric.csv', 'count_in_utc_window': metric_count},
    {'file':'log.csv',    'count_in_utc_window': log_count},
    {'file':'trace.csv',  'count_in_utc_window': trace_count},
    {'file':'error_logs.csv','count_in_utc_window': error_count},
])

# Return compact results (counts + up to 10 sample rows per file)
window_counts, metric_samples, log_samples, trace_samples, error_samples

```
Out[5]:
```
Summary for UTC window 2021-03-23 06:00:00 → 2021-03-23 06:30:00 (inclusive):

- metric.csv
  - Row count in that UTC window: 21,913
  - Up to 10 sample rows (timestamps shown as 2021-03-23 14:00:00+0800, which equals 06:00 UTC): samples include IG01 metrics (JVM HeapMemoryMax, HeapMemoryUsed, NoHeapMemoryUsed, JVM CPULoad, ThreadCount), IG02, and MG01 entries. (Full sample rows are available in metric_samples.)

- log.csv
  - Row count in that UTC window: 467
  - Up to 10 sample rows (timestamps shown as 2021-03-23 14:00:00+0800): samples include Tomcat01–Tomcat04 log.row_count, log.error_count, log.gc_pause_max, and apache01/apache02 log.row_count. (Full sample rows are available in log_samples.)

- trace.csv
  - Row count in that UTC window: 0
  - No sample rows (trace_samples is empty).

- error_logs.csv
  - Row count in that UTC window: 0
  - No sample rows (error_samples is empty).

Notes:
- The UTC window filtering was inclusive of both endpoints.
- Displayed sample timestamps were converted to Asia/Shanghai (UTC+8) for readability (14:00+0800 == 06:00 UTC).

The original code execution output of IPython Kernel is also provided below for reference:

(             file  count_in_utc_window
0      metric.csv                21913
1         log.csv                  467
2       trace.csv                    0
3  error_logs.csv                    0,                   timestamp cmdb_id                                           kpi_name         value
0  2021-03-23 14:00:00+0800    IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  2021-03-23 14:00:00+0800    IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  6.777046e+08
2  2021-03-23 14:00:00+0800    IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.867203e+07
3  2021-03-23 14:00:00+0800    IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  4.699000e-01
4  2021-03-23 14:00:00+0800    IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.500000e+01
5  2021-03-23 14:00:00+0800    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1.700000e+01
6  2021-03-23 14:00:00+0800    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  1.000000e+00
7  2021-03-23 14:00:00+0800    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  1.470000e+02
8  2021-03-23 14:00:00+0800    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  0.000000e+00
9  2021-03-23 14:00:00+0800    IG01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.838620e+01,                   timestamp   cmdb_id          log_name   value
0  2021-03-23 14:00:00+0800  Tomcat01   log.error_count    0.00
1  2021-03-23 14:00:00+0800  Tomcat01     log.row_count  424.00
2  2021-03-23 14:00:00+0800  Tomcat02   log.error_count    0.00
3  2021-03-23 14:00:00+0800  Tomcat02     log.row_count  414.00
4  2021-03-23 14:00:00+0800  Tomcat03   log.error_count    0.00
5  2021-03-23 14:00:00+0800  Tomcat03  log.gc_pause_max    0.08
6  2021-03-23 14:00:00+0800  Tomcat03     log.row_count  449.00
7  2021-03-23 14:00:00+0800  Tomcat04   log.error_count    0.00
8  2021-03-23 14:00:00+0800  Tomcat04  log.gc_pause_max    0.09
9  2021-03-23 14:00:00+0800  Tomcat04     log.row_count  414.00, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing metric_df if present, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp is integer
metric_df['timestamp'] = metric_df['timestamp'].astype('int64')

# Target components
targets = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Restrict to target components (full series)
metrics_targets = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

# Timezone (UTC+8) per rule
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95, count, min, max per (cmdb_id, kpi_name) using full available series
grouped = metrics_targets.groupby(['cmdb_id','kpi_name'])['value']
global_stats = grouped.agg(
    count='count',
    min_value='min',
    p95=lambda x: float(x.quantile(0.95)),
    max_value='max'
).reset_index()

# 2) Filter metric rows to the incident window (inclusive) in UTC+8
start_ts = int(pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz).timestamp())

window_df = metrics_targets[(metrics_targets['timestamp'] >= start_ts) & (metrics_targets['timestamp'] <= end_ts)].copy()

# 3) Join the filtered rows with global P95 per group
# Merge p95 and other global stats into window rows
window_with_stats = window_df.merge(global_stats, on=['cmdb_id','kpi_name'], how='left', suffixes=('','_global'))

# For each group in the window compute desired aggregates
def compute_group_window_aggregates(df):
    # df is grouped window rows for one (cmdb_id,kpi_name)
    cnt = int(df.shape[0])
    p95 = float(df['p95'].iloc[0]) if 'p95' in df.columns else np.nan
    exceed_mask = df['value'] >= p95
    exceed_count = int(exceed_mask.sum())
    if exceed_count > 0:
        earliest_exceed_ts = int(df.loc[exceed_mask, 'timestamp'].min())
        # format to Asia/Shanghai tz string
        earliest_exceed_ts_iso = pd.to_datetime(earliest_exceed_ts, unit='s').tz_localize(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        earliest_exceed_ts_iso = pd.NA
    max_value_in_window = float(df['value'].max())
    return pd.Series({
        'count_window': cnt,
        'exceed_count': exceed_count,
        'earliest_exceed_ts': earliest_exceed_ts_iso,
        'max_value_in_window': max_value_in_window,
        'p95': p95
    })

if window_with_stats.shape[0] == 0:
    # No rows in window at all for targets: produce empty result frame with requested columns
    result_df = pd.DataFrame(columns=['cmdb_id','kpi_name','count_window','exceed_count','earliest_exceed_ts','max_value_in_window','p95'])
else:
    agg_window = window_with_stats.groupby(['cmdb_id','kpi_name']).apply(compute_group_window_aggregates).reset_index()

    # 4) Decide if any group has exceed_count>0
    total_exceeds = int(agg_window['exceed_count'].sum())

    if total_exceeds > 0:
        # Return up to top 20 groups sorted by exceed_count desc
        result_df = agg_window.sort_values(['exceed_count','count_window','max_value_in_window'], ascending=[False,False,False]).head(20)
    else:
        # No exceedances: compute deviation ratio = (max_value_in_window - p95)/p95
        # Handle p95 == 0: set ratio to inf if max>0 else 0
        def dev_ratio(row):
            p = row['p95']
            m = row['max_value_in_window']
            if pd.isna(p) or p == 0:
                if m == 0:
                    return 0.0
                else:
                    return np.inf
            return (m - p) / p
        agg_window['deviation_ratio'] = agg_window.apply(dev_ratio, axis=1)
        result_df = agg_window.sort_values('deviation_ratio', ascending=False).head(20)

# Keep output compact: selected columns and ensure types concise
# If deviation_ratio exists include it; else drop extra internals
cols_standard = ['cmdb_id','kpi_name','count_window','exceed_count','earliest_exceed_ts','max_value_in_window','p95']
if 'deviation_ratio' in result_df.columns:
    display_cols = ['cmdb_id','kpi_name','count_window','exceed_count','earliest_exceed_ts','max_value_in_window','p95','deviation_ratio']
else:
    display_cols = cols_standard

# Final compact result
result_df = result_df[display_cols].reset_index(drop=True)

# Display the compact result
result_df

```
Out[6]:
```
```python
# Build a concise plain-English summary from the last run's result_df (top groups with exceedances)
# The final displayed object must be a variable (summary).

def _fmt_num(x):
    try:
        x = float(x)
    except Exception:
        return str(x)
    if pd.isna(x):
        return 'N/A'
    if abs(x) >= 1e6:
        return f"{int(x):,}"
    if abs(x - int(x)) < 1e-6:
        return f"{int(x)}"
    return f"{x:.3f}"

import pandas as pd

try:
    result_df
except NameError:
    summary = "No result_df found in the environment to summarize."
else:
    # Top examples (up to 5)
    top5 = result_df.head(5)
    lines = []
    lines.append("Metric analysis summary for candidate components (full-series P95 vs. 2021-03-23 14:00–14:30+08 window):")
    lines.append("")
    total_groups = result_df.shape[0]
    total_exceeds = int(result_df['exceed_count'].sum()) if 'exceed_count' in result_df.columns else 0
    lines.append(f"- Groups returned (top {min(20,total_groups)} shown): {total_groups}")
    lines.append(f"- Total exceedance events across these groups (sum of exceed_count): {total_exceeds}")
    lines.append("")
    lines.append("Top example groups (cmdb_id | kpi_name | count_window | exceed_count | earliest_exceed_ts | max_in_window | p95):")
    for r in top5.itertuples(index=False):
        cmdb = getattr(r, 'cmdb_id')
        kpi = getattr(r, 'kpi_name')
        cnt = getattr(r, 'count_window')
        exc = getattr(r, 'exceed_count')
        ets = getattr(r, 'earliest_exceed_ts')
        mval = getattr(r, 'max_value_in_window')
        p95 = getattr(r, 'p95')
        lines.append(f"  • {cmdb} | {kpi} | {cnt} | {exc} | {ets} | max={_fmt_num(mval)} | p95={_fmt_num(p95)}")
    lines.append("")
    lines.append("Interpretation:")
    lines.append("- Many (top) metric groups exceeded their global P95 during the incident window. Exceedances often occurred at the window start (earliest_exceed_ts = 2021-03-23 06:00:00+0800).")
    lines.append("- Notable categories include JVM heap/NoHeap memory maxima across IG/MG hosts and Tomcat request/error and thread-related KPIs.")
    lines.append("- These groups should be prioritized for root-cause investigation (JVM memory and Tomcat request/thread KPIs).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_window  exceed_count        earliest_exceed_ts  max_value_in_window           p95
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31            31  2021-03-23 06:00:00+0800         2.040136e+09  2.040136e+09
1       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31            31  2021-03-23 06:00:00+0800         2.040136e+09  2.040136e+09
2       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31            31  2021-03-23 06:00:00+0800         2.040136e+09  2.040136e+09
3       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31            31  2021-03-23 06:00:00+0800         2.040136e+09  2.040136e+09
4       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31            31  2021-03-23 06:00:00+0800         5.867203e+07  5.867203e+07
5       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31            31  2021-03-23 06:00:00+0800         5.854114e+07  5.854114e+07
6   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31            31  2021-03-23 06:00:00+0800         4.016560e+05  4.016560e+05
7   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31            31  2021-03-23 06:00:00+0800         3.978080e+05  3.978080e+05
8   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31            31  2021-03-23 06:00:00+0800         3.906190e+05  3.906190e+05
9   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31            31  2021-03-23 06:00:00+0800         2.304870e+05  2.304870e+05
10  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31            31  2021-03-23 06:00:00+0800         6.210000e+02  6.210000e+02
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31            31  2021-03-23 06:00:00+0800         5.600000e+02  5.600000e+02
12  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31            31  2021-03-23 06:00:00+0800         5.140000e+02  5.140000e+02
13  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31            31  2021-03-23 06:00:00+0800         5.000000e+02  5.000000e+02
14  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31            31  2021-03-23 06:00:00+0800         5.000000e+02  5.000000e+02
15  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31            31  2021-03-23 06:00:00+0800         5.000000e+02  5.000000e+02
16  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31            31  2021-03-23 06:00:00+0800         5.000000e+02  5.000000e+02
17  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31            31  2021-03-23 06:00:00+0800         1.380000e+02  1.380000e+02
18  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31            31  2021-03-23 06:00:00+0800         1.000000e+02  1.000000e+02
19  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31            31  2021-03-23 06:00:00+0800         1.000000e+02  1.000000e+02```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp is integer seconds
metric_df['timestamp'] = metric_df['timestamp'].astype('int64')

# Candidate components
targets = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Restrict to target components (full series) per rule: compute global thresholds BEFORE any window filtering
metrics_targets = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

# Timezone (UTC+8) per rule
tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95 and basic stats per (cmdb_id, kpi_name) using full available series
grouped_full = metrics_targets.groupby(['cmdb_id','kpi_name'])['value']
global_stats = grouped_full.agg(
    count_full='count',
    min_full='min',
    p95=lambda x: float(x.quantile(0.95)),
    max_full='max'
).reset_index()

# 2) Filter rows to the incident window (inclusive) in UTC+8
start_ts = int(pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz).timestamp())

window_df = metrics_targets[(metrics_targets['timestamp'] >= start_ts) & (metrics_targets['timestamp'] <= end_ts)].copy()

# Helper to format epoch seconds into Asia/Shanghai ISO strings
def fmt_ts_series(s):
    return pd.to_datetime(s, unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# If there are no window rows, produce empty result frames with requested columns
if window_df.empty:
    kpi_level = pd.DataFrame(columns=['cmdb_id','kpi_name','count_window','exceed_count',
                                      'earliest_exceed_ts','max_value_in_window','p95','deviation_pct'])
    comp_level = pd.DataFrame(columns=['cmdb_id','num_faulty_kpis','sum_exceed_count','max_deviation_pct','earliest_exceed_ts_component'])
else:
    # 3) Join window rows with global p95
    win = window_df.merge(global_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute per-group window aggregates
    def agg_group(df):
        cnt = int(df.shape[0])
        p95 = float(df['p95'].iloc[0])
        max_val = float(df['value'].max())
        exceed_mask = df['value'] >= p95
        exc_cnt = int(exceed_mask.sum())
        if exc_cnt > 0:
            earliest_exc_ts_num = int(df.loc[exceed_mask, 'timestamp'].min())
            earliest_exc_ts_iso = pd.to_datetime(earliest_exc_ts_num, unit='s').tz_localize(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        else:
            earliest_exc_ts_num = pd.NA
            earliest_exc_ts_iso = pd.NA
        # deviation_pct handling: avoid division by zero
        if p95 == 0:
            if max_val == 0:
                dev_pct = 0.0
            else:
                dev_pct = np.inf
        else:
            dev_pct = (max_val - p95) / p95 * 100.0
        return pd.Series({
            'count_window': cnt,
            'exceed_count': exc_cnt,
            'earliest_exceed_ts': earliest_exc_ts_iso,
            'earliest_exceed_ts_num': earliest_exc_ts_num,
            'max_value_in_window': max_val,
            'p95': p95,
            'deviation_pct': dev_pct
        })

    agg_window = win.groupby(['cmdb_id','kpi_name']).apply(agg_group).reset_index()

    # 4) KPI-level: sort by deviation_pct desc and return up to top 20, with requested columns
    kpi_level = agg_window.copy()
    # Replace possible inf display with large number for sorting but keep inf in output
    kpi_level = kpi_level.sort_values('deviation_pct', ascending=False).head(20)
    kpi_level = kpi_level[['cmdb_id','kpi_name','count_window','exceed_count','earliest_exceed_ts','max_value_in_window','p95','deviation_pct']].reset_index(drop=True)

    # Component-level aggregation
    comp_agg = agg_window.copy()
    # num_faulty_kpis: count of kpi groups with exceed_count>0
    comp_level = comp_agg.groupby('cmdb_id').apply(lambda df: pd.Series({
        'num_faulty_kpis': int((df['exceed_count']>0).sum()),
        'sum_exceed_count': int(df['exceed_count'].sum()),
        'max_deviation_pct': float(df['deviation_pct'].replace([np.inf], np.nan).max()) if (df['deviation_pct'].replace([np.inf], np.nan).max() is not np.nan) else np.inf,
        'earliest_exceed_ts_component_num': (int(df.loc[df['earliest_exceed_ts_num'].notna(),'earliest_exceed_ts_num'].min()) 
                                             if df['earliest_exceed_ts_num'].notna().any() else pd.NA)
    })).reset_index()

    # Format earliest_exceed_ts_component to ISO string
    def fmt_comp_ts(x):
        if pd.isna(x) or x is pd.NA:
            return pd.NA
        return pd.to_datetime(int(x), unit='s').tz_localize(tz).strftime('%Y-%m-%d %H:%M:%S%z')

    comp_level['earliest_exceed_ts_component'] = comp_level['earliest_exceed_ts_component_num'].apply(fmt_comp_ts)
    comp_level = comp_level[['cmdb_id','num_faulty_kpis','sum_exceed_count','max_deviation_pct','earliest_exceed_ts_component']]

    # Sort components by sum_exceed_count desc and keep up to 20
    comp_level = comp_level.sort_values('sum_exceed_count', ascending=False).head(20).reset_index(drop=True)

# Display compact results: kpi_level (top 20) and comp_level (top 20)
kpi_level, comp_level

```
Out[7]:
```
```python
summary = (
    "Summary of metric analysis for candidate components (window 2021-03-23 14:00–14:30+08):\n\n"
    "KPI-level findings (top groups by deviation_pct):\n"
    "- Several KPI groups show full-window exceedances versus their global P95. Notable top KPIs:\n"
    "  • Mysql02 / Mysql-MySQL_3306_Innodb data pending writes — count_window=29, exceed_count=29, earliest_exceed_ts=2021-03-23 06:01:00+0800, max=1, p95=0 (deviation=inf)\n"
    "  • Mysql02 / Mysql-MySQL_3306_Innodb os log pending writes — same pattern as above (p95=0 → infinite deviation)\n"
    "  • Multiple hosts (Tomcat02, MG02, MG01, Tomcat03, Tomcat04, Tomcat01) show network/TCP and disk/dsk metrics with large deviation_pct (e.g., TCP CLOSE-WAIT, CPU Wio, DSKRead/RTps) and non-trivial exceed_counts.\n"
    "  • Many Tomcat request/error/thread KPIs exceeded their P95 in-window (Request MaxTime, ErrorCount, MaxThreads, CurrentThreads).\n\n"
    "Component-level summary (ranked by total exceed count):\n"
    "- Mysql01 and Mysql02 top the list by sum_exceed_count (Mysql01: sum_exceed_count≈1960, Mysql02≈1786). Both report many KPI groups with exceed_count>0.\n"
    "- Tomcat clusters (Tomcat02, Tomcat03, Tomcat04, Tomcat01) also show large numbers of faulty KPIs (num_faulty_kpis ≈120+) and high sum_exceed_count.\n"
    "- MG01 and MG02 show elevated disk and network deviations as well.\n"
    "- Earliest exceedances for components generally occur at 2021-03-23 06:00:00+0800 (the start of the window).\n\n"
    "Key interpretations and actionables:\n"
    "- MySQL components (Mysql01/Mysql02) and Tomcat instances are the highest-priority suspects: they have the most exceeded KPIs and the largest aggregate exceed counts.\n"
    "- Some KPI P95 baselines are zero (p95=0), producing infinite deviation_pct for even small nonzero values — these represent categorical/rare events (e.g., pending-writes, TCP CLOSE-WAIT) and should be triaged separately.\n"
    "- Also investigate: JVM memory maxima on IG/MG hosts, TCP CLOSE-WAIT/FIN-WAIT states, CPU wait (Wio), and local disk read/rtps spikes on Tomcat/MG hosts.\n\n"
    "Recommended next steps:\n"
    "1) Prioritize MySQL and Tomcat hosts for root-cause debugging (check MySQL pending writes, I/O, locks; check Tomcat request queues/threads and backend connectivity).\n"
    "2) Investigate network/TCP CLOSE-WAIT/FIN-WAIT and disk I/O anomalies on MG/Tomcat hosts.\n"
    "3) For metrics with p95=0, examine raw events/logs to determine whether these are real intermittent faults or noisy/boolean signals that need different handling.\n\n"
    "If you want, I can produce the full top-20 KPI table or expand the component-level drilldowns with exact KPI lists and timestamps."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count_window  exceed_count        earliest_exceed_ts  max_value_in_window         p95  deviation_pct
0    Mysql02        Mysql-MySQL_3306_Innodb data pending writes            29            29  2021-03-23 06:01:00+0800             1.000000    0.000000            inf
1    Mysql02      Mysql-MySQL_3306_Innodb os log pending writes            29            29  2021-03-23 06:01:00+0800             1.000000    0.000000            inf
2   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31            31  2021-03-23 06:00:00+0800             1.000000    0.000000            inf
3       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31            31  2021-03-23 06:00:00+0800             1.000000    0.000000            inf
4       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31            31  2021-03-23 06:00:00+0800             1.000000    0.000000            inf
5    Mysql01        Mysql-MySQL_3306_Innodb data pending writes            29            29  2021-03-23 06:01:00+0800             1.000000    0.000000            inf
6   Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31             2  2021-03-23 06:21:00+0800             6.000000    0.500000    1100.000000
7       MG02                             OSLinux-CPU_CPU_CPUWio            30             2  2021-03-23 06:09:00+0800             3.961100    0.555420     613.172014
8   Tomcat04                             OSLinux-CPU_CPU_CPUWio            31             2  2021-03-23 06:10:00+0800             2.100800    0.447216     369.750635
9   Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuidle            16             1  2021-03-23 06:12:00+0800            20.372200    5.093050     300.000000
10  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            15             1  2021-03-23 06:28:00+0800             0.133333    0.040000     233.333333
11  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15             1  2021-03-23 06:14:00+0800             0.200000    0.060000     233.333333
12  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            15             1  2021-03-23 06:01:00+0800             0.133333    0.040000     233.333333
13  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15             1  2021-03-23 06:26:00+0800             0.200000    0.060000     233.333333
14  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps            15             1  2021-03-23 06:28:00+0800             0.016667    0.005000     233.333333
15  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps            15             1  2021-03-23 06:01:00+0800             0.016667    0.005000     233.333333
16      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            15             1  2021-03-23 06:10:00+0800          1479.066667  448.853333     229.521150
17      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15             1  2021-03-23 06:10:00+0800            10.900000    3.480000     213.218391
18  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31             4  2021-03-23 06:25:00+0800             3.000000    1.000000     200.000000
19  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31             2  2021-03-23 06:02:00+0800            22.000000    8.500000     158.823529,      cmdb_id  num_faulty_kpis  sum_exceed_count  max_deviation_pct earliest_exceed_ts_component
0    Mysql01            137.0            1960.0          87.500000     2021-03-23 06:00:00+0800
1    Mysql02            137.0            1786.0           5.882353     2021-03-23 06:00:00+0800
2   Tomcat02            122.0            1221.0          66.666667     2021-03-23 06:00:00+0800
3   Tomcat03            123.0            1207.0        1100.000000     2021-03-23 06:00:00+0800
4   Tomcat04            122.0            1182.0         369.750635     2021-03-23 06:00:00+0800
5   Tomcat01            120.0            1153.0         233.333333     2021-03-23 06:00:00+0800
6       MG01            104.0             791.0         127.272727     2021-03-23 06:00:00+0800
7       MG02            106.0             776.0         613.172014     2021-03-23 06:00:00+0800
8    Redis02             76.0             628.0         110.526316     2021-03-23 06:00:00+0800
9       IG01             18.0             130.0          53.846154     2021-03-23 06:00:00+0800
10      IG02             18.0             127.0          20.911318     2021-03-23 06:00:00+0800
11   Redis01             11.0              83.0           0.000000     2021-03-23 06:00:00+0800
12  apache01             11.0              81.0          64.713018     2021-03-23 06:00:00+0800
13  apache02             11.0              48.0           0.933799     2021-03-23 06:01:00+0800)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse metric_df from the environment if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp is integer seconds
metric_df['timestamp'] = metric_df['timestamp'].astype('int64')

# UTC+8 timezone
tz = pytz.timezone('Asia/Shanghai')

# Target components and window (UTC+8)
components = ['Mysql01', 'Mysql02']
start_ts = int(pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 14:05:00').tz_localize(tz).timestamp())

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (before any window filtering)
global_stats = (
    metric_df[metric_df['cmdb_id'].isin(components)]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count_full='count',
         min_full='min',
         p95=lambda x: float(x.quantile(0.95)),
         max_full='max')
)

# 2) Filter metric rows to the incident sub-window (inclusive) for the two components
window_rows = metric_df[
    (metric_df['cmdb_id'].isin(components)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Merge global p95 into window rows
if not window_rows.empty:
    win = window_rows.merge(global_stats[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
else:
    win = window_rows.copy()
    win['p95'] = np.nan

# 3) Prepare sample rows (up to 20) sorted by timestamp ascending, format timestamp to UTC+8 ISO
sample_rows = win[['timestamp','cmdb_id','kpi_name','value']].sort_values('timestamp', ascending=True).head(20).copy()
if not sample_rows.empty:
    sample_rows['timestamp'] = pd.to_datetime(sample_rows['timestamp'], unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 4) For each component produce earliest exceed event in-window (value >= p95)
#    Return single-row summary per component: earliest_exceed_ts_in_window, kpi_name, value_at_that_ts
summaries = []
for comp in components:
    comp_win = win[win['cmdb_id'] == comp].copy()
    if comp_win.empty:
        summaries.append({'cmdb_id': comp, 'earliest_exceed_ts_in_window': pd.NA, 'kpi_name': pd.NA, 'value_at_ts': pd.NA})
        continue
    # Rows where value >= p95 (note: p95 may be zero)
    exceed = comp_win[comp_win['value'] >= comp_win['p95']].copy()
    if exceed.empty:
        summaries.append({'cmdb_id': comp, 'earliest_exceed_ts_in_window': pd.NA, 'kpi_name': pd.NA, 'value_at_ts': pd.NA})
    else:
        # earliest exceed by timestamp
        idx = exceed['timestamp'].idxmin()
        row = exceed.loc[idx]
        ts_iso = pd.to_datetime(int(row['timestamp']), unit='s').tz_localize(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        summaries.append({'cmdb_id': comp, 'earliest_exceed_ts_in_window': ts_iso, 'kpi_name': row['kpi_name'], 'value_at_ts': row['value']})

comp_summary = pd.DataFrame(summaries)[['cmdb_id','earliest_exceed_ts_in_window','kpi_name','value_at_ts']]

# Output compact results: sample_rows (<=20) and comp_summary (one row per component)
sample_rows, comp_summary

```
Out[8]:
```
# Build a concise plain-English summary string using existing results: sample_rows and comp_summary
try:
    sample_rows
    comp_summary
except NameError:
    summary = "No sample_rows or comp_summary found in the environment."
else:
    n_rows = int(sample_rows.shape[0])
    comps = sorted(sample_rows['cmdb_id'].unique().tolist()) if n_rows>0 else []
    lines = []
    lines.append("Summary of extracted metric rows for Mysql01 and Mysql02")
    lines.append("Window: 2021-03-23 14:00:00+0800 through 2021-03-23 14:05:00+0800 (inclusive).")
    lines.append("")
    lines.append(f"- Returned sample rows (up to 20): {n_rows}")
    lines.append(f"- Components present in samples: {', '.join(comps) if comps else 'none'}")
    lines.append("")
    lines.append("Per-component earliest exceedance in-window (value >= group's global P95):")
    for r in comp_summary.itertuples(index=False):
        comp = r.cmdb_id
        ets = r.earliest_exceed_ts_in_window if pd.notna(r.earliest_exceed_ts_in_window) else "null"
        kpi = r.kpi_name if pd.notna(r.kpi_name) else "null"
        val = r.value_at_ts if pd.notna(r.value_at_ts) else "null"
        lines.append(f"  • {comp}: earliest_exceed_ts_in_window = {ets} | KPI = {kpi} | value = {val}")
    lines.append("")
    lines.append("Notes:")
    lines.append("- Timestamps displayed are in Asia/Shanghai (UTC+8).")
    lines.append("- Full sample rows are available in the variable `sample_rows` (up to 20 rows).")
    lines.append("- Per-component summary is available in the variable `comp_summary`.")
    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  cmdb_id                                           kpi_name     value
0   2021-03-23 06:00:00+0800  Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld    1.0000
1   2021-03-23 06:00:00+0800  Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld    0.0000
2   2021-03-23 06:00:00+0800  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...   27.0000
3   2021-03-23 06:00:00+0800  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    1.0000
4   2021-03-23 06:00:00+0800  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...    4.0000
5   2021-03-23 06:00:00+0800  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.0000
6   2021-03-23 06:00:00+0800  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRea...  727.0000
7   2021-03-23 06:00:00+0800  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKTps   27.0000
8   2021-03-23 06:00:00+0800  Mysql01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc   98.0000
9   2021-03-23 06:00:00+0800  Mysql01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc   91.8757
10  2021-03-23 06:00:00+0800  Mysql02         Mysql-MySQL_3306_GetConnectedStateOfMysqld    1.0000
11  2021-03-23 06:00:00+0800  Mysql02           Mysql-MySQL_3306_GetResponseTimeOfMysqld    0.0000
12  2021-03-23 06:00:00+0800  Mysql02                    OSLinux-CPU_CPU-0_SingleCpuUtil   18.1803
13  2021-03-23 06:00:00+0800  Mysql02                    OSLinux-CPU_CPU-1_SingleCpuUtil   19.0074
14  2021-03-23 06:00:00+0800  Mysql02                    OSLinux-CPU_CPU-2_SingleCpuUtil   15.3833
15  2021-03-23 06:00:00+0800  Mysql02                    OSLinux-CPU_CPU-3_SingleCpuUtil   15.0868
16  2021-03-23 06:00:00+0800  Mysql02                         OSLinux-CPU_CPU_CPUCpuUtil   16.1148
34  2021-03-23 06:01:00+0800  Mysql01                  Mysql-MySQL_3306_Com Update Multi    0.0000
35  2021-03-23 06:01:00+0800  Mysql01                       Mysql-MySQL_3306_Connections    0.1000
36  2021-03-23 06:01:00+0800  Mysql01           Mysql-MySQL_3306_Created Tmp Disk Tables    0.0167,    cmdb_id earliest_exceed_ts_in_window                                    kpi_name  value_at_ts
0  Mysql01     2021-03-23 06:00:00+0800  Mysql-MySQL_3306_GetConnectedStateOfMysqld          1.0
1  Mysql02     2021-03-23 06:00:00+0800  Mysql-MySQL_3306_GetConnectedStateOfMysqld          1.0)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing dataframes if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure timestamp is integer seconds
metric_df['timestamp'] = metric_df['timestamp'].astype('int64')
log_df['timestamp'] = log_df['timestamp'].astype('int64')

# UTC+8 timezone for conversions per rule
tz = pytz.timezone('Asia/Shanghai')

# Target MySQL components
components = ['Mysql01', 'Mysql02']

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full available series BEFORE any window filtering
global_stats_mysql = (
    metric_df[metric_df['cmdb_id'].isin(components)]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count_full='count',
         min_full='min',
         p95=lambda x: float(x.quantile(0.95)),
         max_full='max')
)

# Incident window (UTC+8) inclusive
start_ts = int(pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz).timestamp())

# 2) Filter metric rows to the incident window for the two components
window_rows = metric_df[
    (metric_df['cmdb_id'].isin(components)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Merge global p95 into window rows
if not window_rows.empty:
    win = window_rows.merge(global_stats_mysql[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
else:
    win = window_rows.copy()
    win['p95'] = np.nan

# A) KPI exceed summary for groups present in the window
if win.empty:
    kpi_exceed_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_window','exceed_count',
        'earliest_exceed_ts_in_window','value_at_earliest_exceed','p95'
    ])
else:
    # Helper to compute per-group aggregates
    def agg_kpi(df):
        cnt = int(df.shape[0])
        p95 = float(df['p95'].iloc[0]) if not pd.isna(df['p95'].iloc[0]) else np.nan
        exceed_mask = df['value'] >= p95
        exc_cnt = int(exceed_mask.sum())
        if exc_cnt > 0:
            # earliest exceed timestamp (numeric) and value at that timestamp (first occurrence)
            exc_rows = df.loc[exceed_mask].sort_values('timestamp', ascending=True)
            earliest_ts_num = int(exc_rows['timestamp'].iloc[0])
            earliest_ts_iso = pd.to_datetime(earliest_ts_num, unit='s').tz_localize(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            val_at_earliest = float(exc_rows['value'].iloc[0])
        else:
            earliest_ts_num = pd.NA
            earliest_ts_iso = pd.NA
            val_at_earliest = pd.NA
        return pd.Series({
            'count_window': cnt,
            'exceed_count': exc_cnt,
            'earliest_exceed_ts_num': earliest_ts_num,
            'earliest_exceed_ts_in_window': earliest_ts_iso,
            'value_at_earliest_exceed': val_at_earliest,
            'p95': p95
        })

    kpi_agg = win.groupby(['cmdb_id','kpi_name']).apply(agg_kpi).reset_index()

    # Filter to exceed_count>0, sort by earliest_exceed_ts ascending then exceed_count desc, limit 20
    kpi_exceed_summary = kpi_agg[kpi_agg['exceed_count'] > 0].copy()
    if not kpi_exceed_summary.empty:
        # For sorting, ensure earliest_exceed_ts_num present; rows with NA put at end
        kpi_exceed_summary['earliest_exceed_ts_num_sort'] = kpi_exceed_summary['earliest_exceed_ts_num'].fillna(10**12)
        kpi_exceed_summary = kpi_exceed_summary.sort_values(
            ['earliest_exceed_ts_num_sort', 'exceed_count'], ascending=[True, False]
        ).head(20)
        kpi_exceed_summary = kpi_exceed_summary[[
            'cmdb_id','kpi_name','count_window','exceed_count',
            'earliest_exceed_ts_in_window','value_at_earliest_exceed','p95'
        ]].reset_index(drop=True)
    else:
        # No exceedances: return empty frame with requested columns
        kpi_exceed_summary = kpi_exceed_summary[[
            'cmdb_id','kpi_name','count_window','exceed_count',
            'earliest_exceed_ts_in_window','value_at_earliest_exceed','p95'
        ]]

# B) Log metric samples and aggregated counts for the same components and window
log_window = log_df[
    (log_df['cmdb_id'].isin(components)) &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

if log_window.empty:
    log_samples = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
    log_counts = pd.DataFrame(columns=['cmdb_id','count_in_window'])
else:
    # Prepare up to 20 sample rows sorted by timestamp ascending
    log_samples = log_window[['timestamp','cmdb_id','log_name','value']].sort_values('timestamp', ascending=True).head(20).copy()
    log_samples['timestamp'] = pd.to_datetime(log_samples['timestamp'], unit='s').dt.tz_localize(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    # Aggregated counts per cmdb_id in-window
    log_counts = log_window.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'count_in_window'})

# Return compact results as variables for IPython display
kpi_exceed_summary, log_samples, log_counts

```
Out[9]:
```
Summary (Mysql01 & Mysql02, window 2021-03-23 14:00–14:30+08):

Key findings — KPI exceed summary
- I returned up to 20 MySQL KPI groups that show exceedances (value >= global P95) during the window.
- Many KPI groups for both Mysql01 and Mysql02 exceeded their global P95 right at the window start: earliest exceed timestamps are predominantly 2021-03-23 06:00:00+0800 (the start of the window).
- Notable KPI groups (examples from the top results):
  - Mysql-MySQL_3306_GetConnectedStateOfMysqld (Mysql01 and Mysql02): count_window=30, exceed_count=30, earliest=2021-03-23 06:00:00+0800, value=1, p95=1.
  - Mysql-MySQL_3306_GetResponseTimeOfMysqld (both): count_window=30, exceed_count=30, earliest=2021-03-23 06:00:00+0800, value=0, p95=0.
  - OSLinux memory/disk KPIs on Mysql01 (MEMUsedMemPerc, DSKTps, DSKRead for sda/sdb/sdc): multiple groups with exceed_count>0, earliest exceed at 06:00:00+0800, and notable high values (e.g., sdc_DSKRead large in the sample rows).
  - Binlog / statement-cache KPIs on Mysql01 show full-window exceedances (e.g., Binlog stmt cache use = 360, exceed_count=29).
- A number of KPIs have p95 == 0; for such metrics any non-zero value registers as an exceedance (leading to infinite or nominally extreme deviation_pct). Treat these boolean/rare-event metrics separately in triage.

Log evidence (in-window)
- Extracted up to 20 log rows in-window; counts per component in-window:
  - Mysql01: 2 log rows
  - Mysql02: 4 log rows
- Sample log rows show mostly log.row_count entries and log.error_count = 0 (no clear error entries in these sampled log rows). No strong error-log evidence in the sampled logs to contradict the metric indications.

Interpretation and recommended next steps
- Both Mysql01 and Mysql02 show immediate, broad metric exceedances at the start of the incident window. Primary suspects: MySQL-related internal metrics (pending writes/binlog/cache), memory usage, and local disk activity (DSKRead/DSKTps).
- Logs sampled during the window show few error entries (log.error_count mostly 0), so metric anomalies may indicate resource saturation (I/O or memory) or MySQL internal contention rather than explicit application-level errors.
- Triage recommendations:
  1. Investigate MySQL pending-writes / binlog and statement-cache metrics (Mysql01 shows many such KPIs exceeding baseline).
  2. Check disk performance on MySQL hosts (sdc/sda/sdb DSKRead/RTps/DSKTps spikes) and underlying storage subsystem and I/O queue.
  3. Correlate with system-level traces/logs (slow queries, lock contention, IO wait) on Mysql01/Mysql02 around 2021-03-23 06:00:00+0800.
  4. For KPIs with p95=0 (boolean/rare events like pending-writes or TCP CLOSE-WAIT), inspect raw events/logs to determine real impact vs noise.

If you want, I can:
- Show the full top-20 KPI table (exact values) for Mysql01/Mysql02, or
- Expand log sampling (more rows) and search logs for any error messages or slow-query traces around the earliest exceed timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  count_window  exceed_count earliest_exceed_ts_in_window  value_at_earliest_exceed       p95
0   Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld            30            30     2021-03-23 06:00:00+0800                    1.0000    1.0000
1   Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld            30            30     2021-03-23 06:00:00+0800                    0.0000    0.0000
2   Mysql02         Mysql-MySQL_3306_GetConnectedStateOfMysqld            30            30     2021-03-23 06:00:00+0800                    1.0000    1.0000
3   Mysql02           Mysql-MySQL_3306_GetResponseTimeOfMysqld            30            30     2021-03-23 06:00:00+0800                    0.0000    0.0000
4   Mysql01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc            22            22     2021-03-23 06:00:00+0800                   98.0000   98.0000
5   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps            13            13     2021-03-23 06:00:00+0800                    0.0000    0.0000
6   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...            11             7     2021-03-23 06:00:00+0800                    4.0000    4.0000
7   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKTps            13             3     2021-03-23 06:00:00+0800                   27.0000   27.0000
8   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps            12             2     2021-03-23 06:00:00+0800                    1.0000    1.0000
9   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...            10             1     2021-03-23 06:00:00+0800                   27.0000   20.7000
10  Mysql02                    OSLinux-CPU_CPU-0_SingleCpuUtil             9             1     2021-03-23 06:00:00+0800                   18.1803   17.2753
11  Mysql01        Mysql-MySQL_3306_Binlog stmt cache disk use            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
12  Mysql01             Mysql-MySQL_3306_Binlog stmt cache use            29            29     2021-03-23 06:01:00+0800                  360.0000  360.0000
13  Mysql01                  Mysql-MySQL_3306_Com Delete Multi            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
14  Mysql01                 Mysql-MySQL_3306_Com Insert Select            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
15  Mysql01                          Mysql-MySQL_3306_Com Load            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
16  Mysql01                       Mysql-MySQL_3306_Com Replace            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
17  Mysql01                Mysql-MySQL_3306_Com Replace Select            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
18  Mysql01                  Mysql-MySQL_3306_Com Update Multi            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000
19  Mysql01          Mysql-MySQL_3306_CurrentSQLMaxRunningTime            29            29     2021-03-23 06:01:00+0800                    0.0000    0.0000,                     timestamp  cmdb_id         log_name  value
103  2021-03-23 06:07:00+0800  Mysql02  log.error_count    0.0
104  2021-03-23 06:07:00+0800  Mysql02    log.row_count    1.0
149  2021-03-23 06:10:00+0800  Mysql01  log.error_count    0.0
150  2021-03-23 06:10:00+0800  Mysql01    log.row_count    1.0
151  2021-03-23 06:10:00+0800  Mysql02  log.error_count    0.0
152  2021-03-23 06:10:00+0800  Mysql02    log.row_count    1.0,    cmdb_id  count_in_window
0  Mysql01                2
1  Mysql02                4)```
```